In [1]:
# Cell 1: dựng lại bộ công cụ cho notebook mới
from google.cloud import bigquery

PROJECT = "bq-learning-510104"
client = bigquery.Client(project=PROJECT, location="US")  # kết nối Python với BigQuery

def mb(n):
    # đổi byte sang MB; n là None (job chưa xong) thì trả về None, không ép thành 0
    return None if n is None else round(n / 1024 / 1024, 2)

def check(sql):
    # dry run: chỉ ước lượng, không chạy thật
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)  # chỉ tạo cấu hình
    job = client.query(sql, job_config=cfg)                              # ước lượng xảy ra ở dòng này
    print(f"Sẽ quét: {mb(job.total_bytes_processed)} MB")

def stats(sql, max_mb=100):
    # chạy thật, tắt cache, giới hạn bytes billed, in processed và billed
    cfg = bigquery.QueryJobConfig(use_query_cache=False,
                                  maximum_bytes_billed=max_mb * 1024 * 1024)
    job = client.query(sql, job_config=cfg)
    df = job.result().to_dataframe()   # job.result() chờ job xong rồi mới đọc số byte
    print(f"processed: {mb(job.total_bytes_processed)} MB | billed: {mb(job.total_bytes_billed)} MB")
    return df

print("Đã sẵn sàng")

Đã sẵn sàng


In [2]:
# Cell 2
# Mục đích: tự thấy khối except chạy khi có lỗi thật (hôm trước nó viết sai nên chưa từng bị thử)
# get_table chỉ đọc thông tin bảng (metadata), không tạo query job nên không tốn tiền

test_ids = [
    "bq-learning-510104.lab.fact_items_plain",    # bảng thật
    "bq-learning-510104.lab.bang_khong_ton_tai",  # bảng sai, cố ý
]

for table_id in test_ids:                         # mỗi vòng lấy một tên bảng
    try:                                          # thử làm việc này trước
        t = client.get_table(table_id)            # nhờ BigQuery trả thông tin đầy đủ của một bảng
        print("OK  ", table_id, "-", t.num_rows, "dòng")
    except Exception as e:                        # nếu bảng trên lỗi, nhảy xuống đây thay vì dừng cell
        print("LỖI ", table_id, "-", type(e).__name__)   # type(e).__name__ là tên loại lỗi

OK   bq-learning-510104.lab.fact_items_plain - 180771 dòng
LỖI  bq-learning-510104.lab.bang_khong_ton_tai - NotFound


In [3]:
# Cell 3
# Mục đích: kiểm tra đường chạy query thật (client.query → job.result() → to_dataframe) còn hoạt động
#           sau khi protobuf bị hạ từ 7.36.2 xuống 6.33.6 khi cài dbt. Cell 2 mới thử get_table,
#           chưa thử chạy query.
# SELECT 1 không đọc bảng nào nên gần như chắc chắn 0 byte, không tốn tiền

df = stats("SELECT 1 AS x")   # stats chạy thật, in processed/billed, trả về DataFrame
print(df)                     # in DataFrame ra để thấy .to_dataframe() hoạt động

processed: 0.0 MB | billed: 0.0 MB
   x
0  1
